Nama : Aidatul Rosida  
NIM : 2341760029

## Topik
Analisis Risiko Klaim Asuransi Menggunakan Sistem Pakar
dengan Metode Forward Chaining dan Backward Chaining.

Import Library

In [ ]:
import pandas as pd

Load Dataset

In [ ]:
polis = pd.read_csv("Data_Polis.csv")
klaim = pd.read_csv("Data_Klaim.csv")

print("Data Polis:")
print(polis.head())

print("\nData Klaim:")
print(klaim.head())

Data Polis:
  Nomor Polis Plan Code Gender  Tanggal Lahir  Tanggal Efektif Polis Domisili
0    POL-0001     M-003      M       19640811               20140603  JAKARTA
1    POL-0002     M-003      M       19710730               20140603  JAKARTA
2    POL-0003     M-001      M       19790821               20160808  JAKARTA
3    POL-0004     M-003      M       20140724               20160811  JAKARTA
4    POL-0005     M-001      F       19810114               20150828  JAKARTA

Data Klaim:
   Claim ID Nomor Polis Reimburse/Cashless Inpatient/Outpatient ICD Diagnosis  \
0  C-0001-M    POL-0176                  R                   OP           C50   
1  C-0002-M    POL-3288                  R                   OP           C34   
2  C-0003-M    POL-1786                  R                   OP         C18.9   
3  C-0004-M    POL-1786                  R                   OP           C34   
4  C-0005-M    POL-2778                  R                   OP           C50   

                    

Preprocessing Data

In [ ]:
polis.columns = polis.columns.str.lower().str.replace(" ", "_")
klaim.columns = klaim.columns.str.lower().str.replace(" ", "_")

data = pd.merge(polis, klaim, on="nomor_polis")

print("Dataset Gabungan")
print(data.head())

Dataset Gabungan
  nomor_polis plan_code gender  tanggal_lahir  tanggal_efektif_polis domisili  \
0    POL-0003     M-001      M       19790821               20160808  JAKARTA   
1    POL-0006     M-001      F       19551127               20160628  JAKARTA   
2    POL-0006     M-001      F       19551127               20160628  JAKARTA   
3    POL-0010     M-001      F       19570605               20170908  JAKARTA   
4    POL-0010     M-001      F       19570605               20170908  JAKARTA   

   claim_id reimburse/cashless inpatient/outpatient icd_diagnosis  \
0  C-3535-M                  C                   IP         S83.2   
1  C-4498-M                  C                   IP         S46.0   
2  C-4499-M                  C                   IP         T84.7   
3  C-3447-M                  R                   IP         K31.7   
4  C-3448-M                  R                   IP         K31.7   

                                     icd_description status_klaim  \
0           

Transformasi Fitur Menjadi Fakta

In [ ]:
data["kategori_klaim"] = data["nominal_klaim_yang_disetujui"].apply(
    lambda x: "klaim_tinggi" if x > 10000000 else "klaim_rendah"
)

print(data[["nomor_polis","nominal_klaim_yang_disetujui","kategori_klaim"]].head())

  nomor_polis  nominal_klaim_yang_disetujui kategori_klaim
0    POL-0003                    14138163.0   klaim_tinggi
1    POL-0006                   117940836.0   klaim_tinggi
2    POL-0006                    60671660.0   klaim_tinggi
3    POL-0010                     1723747.0   klaim_rendah
4    POL-0010                    63251531.0   klaim_tinggi


Mengambil Fakta dari Dataset

In [ ]:
def ambil_fakta(row):

    fakta = []

    if row["inpatient/outpatient"] == "Inpatient":
        fakta.append("inpatient")
    else:
        fakta.append("outpatient")

    if row["reimburse/cashless"] == "Reimburse":
        fakta.append("reimburse")
    else:
        fakta.append("cashless")

    fakta.append(row["kategori_klaim"])

    return fakta


sample = data.iloc[0]

facts = ambil_fakta(sample)

print("Fakta dari data:")
print(facts)

Fakta dari data:
['outpatient', 'cashless', 'klaim_tinggi']


Definisi Rule Sistem Pakar
Rule dibuat dalam bentuk IF–THEN untuk menentukan kesimpulan
berdasarkan fakta yang tersedia

In [ ]:
rules = [

    {"if": ["inpatient","klaim_tinggi"], "then": "risiko_klaim_tinggi"},

    {"if": ["outpatient","klaim_rendah"], "then": "risiko_klaim_rendah"},

    {"if": ["reimburse","klaim_tinggi"], "then": "perlu_verifikasi"},

    {"if": ["cashless","klaim_tinggi"], "then": "klaim_besar"},

    {"if": ["inpatient","klaim_rendah"], "then": "klaim_normal"}

]

Implementasi Forward Chaining
**teks tebal**
Forward chaining dimulai dari fakta yang tersedia kemudian
mengevaluasi rule untuk menghasilkan kesimpulan baru.

In [ ]:
def forward_chaining(facts, rules):

    hasil = []

    for rule in rules:
        if all(cond in facts for cond in rule["if"]):
            hasil.append(rule["then"])

    return hasil


hasil_forward = forward_chaining(facts, rules)

print("Hasil Forward Chaining:")
print(hasil_forward)

Hasil Forward Chaining:
['klaim_besar']


Uji Forward Chaining pada Beberapa Data

In [ ]:
for i in range(10):

    sample = data.iloc[i]

    facts = ambil_fakta(sample)

    hasil = forward_chaining(facts, rules)

    print("Data ke-", i)
    print("Fakta:", facts)
    print("Kesimpulan:", hasil)
    print()

Data ke- 0
Fakta: ['outpatient', 'cashless', 'klaim_tinggi']
Kesimpulan: ['klaim_besar']

Data ke- 1
Fakta: ['outpatient', 'cashless', 'klaim_tinggi']
Kesimpulan: ['klaim_besar']

Data ke- 2
Fakta: ['outpatient', 'cashless', 'klaim_tinggi']
Kesimpulan: ['klaim_besar']

Data ke- 3
Fakta: ['outpatient', 'cashless', 'klaim_rendah']
Kesimpulan: ['risiko_klaim_rendah']

Data ke- 4
Fakta: ['outpatient', 'cashless', 'klaim_tinggi']
Kesimpulan: ['klaim_besar']

Data ke- 5
Fakta: ['outpatient', 'cashless', 'klaim_tinggi']
Kesimpulan: ['klaim_besar']

Data ke- 6
Fakta: ['outpatient', 'cashless', 'klaim_tinggi']
Kesimpulan: ['klaim_besar']

Data ke- 7
Fakta: ['outpatient', 'cashless', 'klaim_tinggi']
Kesimpulan: ['klaim_besar']

Data ke- 8
Fakta: ['outpatient', 'cashless', 'klaim_rendah']
Kesimpulan: ['risiko_klaim_rendah']

Data ke- 9
Fakta: ['outpatient', 'cashless', 'klaim_tinggi']
Kesimpulan: ['klaim_besar']



**Implementasi Backward Chaining**

Backward chaining dimulai dari goal kemudian menelusuri rule
untuk membuktikan apakah goal dapat dicapai berdasarkan fakta.

In [ ]:
def backward_chaining(goal, facts, rules):

    print("Mencari rule untuk goal:", goal)

    for rule in rules:

        if rule["then"] == goal:

            print("Rule ditemukan:", rule)

            for cond in rule["if"]:

                if cond not in facts:
                    print("Fakta tidak memenuhi:", cond)
                    return False

            print("Semua kondisi terpenuhi")
            return True

    print("Tidak ada rule untuk goal ini")
    return False


goal = "risiko_klaim_tinggi"

hasil = backward_chaining(goal, facts, rules)

print("\nHasil:")
if hasil:
    print("Goal terbukti")
else:
    print("Goal tidak terbukti")

Mencari rule untuk goal: risiko_klaim_tinggi
Rule ditemukan: {'if': ['inpatient', 'klaim_tinggi'], 'then': 'risiko_klaim_tinggi'}
Fakta tidak memenuhi: inpatient

Hasil:
Goal tidak terbukti


Analisis Akhir

Forward chaining menghasilkan kesimpulan berdasarkan fakta yang
tersedia dalam dataset. Sistem akan mengevaluasi setiap rule dan
menjalankan rule yang memenuhi kondisi.

Backward chaining digunakan untuk membuktikan apakah suatu goal
dapat dicapai. Pada beberapa data goal tidak terbukti karena fakta
yang tersedia tidak memenuhi kondisi rule yang dibutuhkan.

Dengan menggunakan kedua metode ini sistem pakar dapat melakukan
reasoning terhadap data klaim asuransi untuk menentukan tingkat
risiko klaim.

**Implementasi Metode teorema bayes**

In [ ]:
# Probabilitas klaim disetujui (PAID)
p_klaim = len(klaim[klaim["status_klaim"] == "PAID"]) / len(klaim)

# Probabilitas klaim besar (> 10 juta)
p_besar = len(klaim[klaim["nominal_klaim_yang_disetujui"] > 10000000]) / len(klaim)

# Likelihood: P(Besar | Klaim)
p_besar_given_klaim = len(
    klaim[(klaim["status_klaim"] == "PAID") &
          (klaim["nominal_klaim_yang_disetujui"] > 10000000)]
) / len(klaim[klaim["status_klaim"] == "PAID"])

# Teorema Bayes
p_klaim_given_besar = (p_besar_given_klaim * p_klaim) / p_besar

print("P(Klaim):", round(p_klaim, 3))
print("P(Besar):", round(p_besar, 3))
print("P(Klaim | Besar):", round(p_klaim_given_besar, 3))
print("Persentase:", round(p_klaim_given_besar * 100, 2), "%")

P(Klaim): 1.0
P(Besar): 0.549
P(Klaim | Besar): 1.0
Persentase: 100.0 %


Berdasarkan perhitungan menggunakan Teorema Bayes, diperoleh nilai P(Klaim | Besar) sebesar 1.0 atau 100%. Hal ini menunjukkan bahwa seluruh klaim dengan nominal besar dalam dataset selalu disetujui (PAID). Kondisi ini terjadi karena seluruh data klaim dalam dataset memiliki status disetujui, sehingga probabilitas klaim disetujui menjadi maksimal.

**Implementasi Metode Certainty Factors**

Perhitungan Certainty Factor untuk menentukan risiko klaim tinggi
berdasarkan evidence:
- Inpatient
- Klaim Tinggi
- Reimburse

In [ ]:
# Ambil 1 data sebagai contoh
sample = klaim.iloc[0]

# Tentukan evidence dari dataset
evidence = []

# Evidence 1: Inpatient / Outpatient
if sample["inpatient/outpatient"] == "IP":
    evidence.append("inpatient")
else:
    evidence.append("outpatient")

# Evidence 2: Metode klaim
if sample["reimburse/cashless"] == "R":
    evidence.append("reimburse")
else:
    evidence.append("cashless")

# Evidence 3: Nominal klaim
if sample["nominal_klaim_yang_disetujui"] > 10000000:
    evidence.append("klaim_tinggi")
else:
    evidence.append("klaim_rendah")

print("Evidence:", evidence)

Evidence: ['outpatient', 'reimburse', 'klaim_tinggi']


In [ ]:
# CF dari pakar (aturan sistem)
cf_pakar = {
    "inpatient": 0.8,
    "outpatient": 0.5,
    "reimburse": 0.6,
    "cashless": 0.7,
    "klaim_tinggi": 1.0,
    "klaim_rendah": 0.4
}

# CF dari user / data
cf_user = {
    "inpatient": 0.6,
    "outpatient": 0.5,
    "reimburse": 0.4,
    "cashless": 0.8,
    "klaim_tinggi": 0.8,
    "klaim_rendah": 0.5
}

In [ ]:
# ===== Hitung CF tiap evidence =====
cf_values = []
for f in fakta:
    cf = cf_pakar[f] * cf_user[f]
    cf_values.append(cf)

print("CF tiap evidence:", cf_values)


# ===== Kombinasi CF =====
cf_total = cf_values[0]

for i in range(1, len(cf_values)):
    cf_total = cf_total + cf_values[i] * (1 - cf_total)

print("CF akhir:", round(cf_total, 3))
print("Persentase:", round(cf_total * 100, 2), "%")

CF tiap evidence: [0.25, 0.24, 0.8]
CF akhir: 0.886
Persentase: 88.6 %


Nilai CF sebesar 88.6% menunjukkan bahwa klaim memiliki tingkat keyakinan tinggi untuk dikategorikan sebagai risiko tinggi.